# 估值异象完整教程：F-Score 与价值投资

## 📚 教学目标
1. 理解**估值异象**的核心：估值指标（E/P, B/M, CF/P）能预测未来收益
2. 掌握 **F-Score** 的 9 个指标及其构建方法（盈利能力、财务状况、经营效率）
3. 在微型数据集（10 只股票）上**手算** F-Score 和分组检验
4. 扩展到 300 只股票 × 60 个月，完成**异象检验**和 Alpha 分析
5. 区分**真异象**（显著 Alpha）和**伪异象**（被已知因子解释）

**参考书**：《因子投资：方法与实践》（石川）第 5.1 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

---

## 1. 什么是估值异象？为什么需要 F-Score？

### 🎯 一个直觉问题

假设你发现一只股票 PE 很低（比如 5 倍），看起来很便宜。但问题是：

- **情况 A**：股价被低估，未来会涨回来 → 买入赚钱
- **情况 B**：公司基本面恶化，PE 低是因为盈利暴跌 → 买入亏钱

**单纯按估值指标选股，无法区分这两种情况！**

### 📖 书中的定义

> 价值因子通常按照估值指标的高低选股。依照这个法则，投资者应尽量选择看起来较便宜的股票，避开那些已经很贵的股票。然而这是事实，却不是真相。一个股票估值之所以便宜，是因为要么价格真的偏离了内在价值，要么本来就不值钱。前者属于真正的价值股，后者可能存在估值陷阱。

### 💡 核心洞察

| 概念 | 含义 |
|------|------|
| 估值因子 | 按 PE/PB/PS 等指标排序选股 |
| 估值陷阱 | 低估值但基本面恶化的股票 |
| F-Score | Piotroski (2000) 提出的 9 指标基本面评分 |
| 价值投资 | 估值因子 + 基本面筛选 = 找到真正便宜的好公司 |

In [ ]:
import sys, os
print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")
try:
    import matplotlib
    print(f"matplotlib: {matplotlib.__version__}")
except ImportError:
    print("❌ matplotlib 未安装! 请运行: !pip install matplotlib seaborn statsmodels scipy")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. F-Score 的 9 个指标

### 📐 F-Score 指标体系

Piotroski (2000) 使用三大类共 9 个指标来度量一家公司的好坏：

#### 盈利能力（4 个指标）
| 指标 | 定义 | 得分条件 |
|------|------|----------|
| ROA | 净利润 / 总资产 | > 0 得 1 分 |
| ΔROA | ROA 同比变化 | > 0 得 1 分 |
| CFO | 经营现金流 / 总资产 | > 0 得 1 分 |
| ACCRUE | 应计利润 = ROA - CFO | < 0 得 1 分（现金流 > 利润） |

#### 财务状况（3 个指标）
| 指标 | 定义 | 得分条件 |
|------|------|----------|
| ΔLEVER | 资产负债率同比变化 | < 0 得 1 分（杠杆下降） |
| ΔLIQUID | 流动比率同比变化 | > 0 得 1 分 |
| EQ_OFFER | 是否增发股票 | 未增发得 1 分 |

#### 经营效率（2 个指标）
| 指标 | 定义 | 得分条件 |
|------|------|----------|
| ΔMARGIN | 毛利率同比变化 | > 0 得 1 分 |
| ΔTURN | 资产周转率同比变化 | > 0 得 1 分 |

**F-Score 总分 = 0~9 分，分值越高说明公司基本面越强**

---

## 3. 微型示例：10 只股票的 F-Score 计算

### 🎯 场景

假设市场上只有 **10 只股票**，我们已知它们的估值指标（B/M）和基本面数据。

我们要：
1. 计算每只股票的 F-Score
2. 按 B/M 和 F-Score 分组
3. 检验 F-Score 能否区分"真价值"和"价值陷阱"

In [ ]:
# ========== 构造 10 只股票的微型数据 ==========
np.random.seed(42)

stocks = ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10']

# 估值指标：B/M（账面市值比）
bm_ratio = np.array([0.8, 0.5, 1.2, 0.3, 1.0, 0.4, 0.7, 1.5, 0.2, 0.9])

# F-Score 的 9 个指标（0 或 1）
# 盈利能力
roa_positive = np.array([1, 0, 1, 0, 1, 0, 1, 1, 0, 1])       # ROA > 0
delta_roa = np.array([1, 0, 1, 0, 0, 0, 1, 1, 0, 0])          # ΔROA > 0
cfo_positive = np.array([1, 0, 1, 0, 1, 0, 1, 1, 0, 1])       # CFO > 0
accrue_negative = np.array([1, 0, 1, 0, 1, 0, 0, 1, 0, 1])    # ACCRUE < 0

# 财务状况
delta_lever = np.array([1, 0, 1, 0, 1, 0, 1, 1, 0, 0])        # ΔLEVER < 0
delta_liquid = np.array([1, 0, 1, 0, 1, 0, 1, 1, 0, 1])       # ΔLIQUID > 0
eq_offer = np.array([1, 1, 1, 0, 1, 0, 1, 1, 0, 1])          # 未增发

# 经营效率
delta_margin = np.array([1, 0, 1, 0, 0, 0, 1, 1, 0, 1])       # ΔMARGIN > 0
delta_turn = np.array([1, 0, 1, 0, 1, 0, 0, 1, 0, 0])         # ΔTURN > 0

# 计算 F-Score
f_score = (roa_positive + delta_roa + cfo_positive + accrue_negative +
           delta_lever + delta_liquid + eq_offer +
           delta_margin + delta_turn)

# 未来收益率（模拟：高 F-Score + 低 B/M → 高收益）
# 真实效应：F-Score 每增加 1 分，未来收益增加 0.5%
gamma_fscore = 0.5
noise = np.random.normal(0, 2, 10)
future_returns = 2 + gamma_fscore * f_score - 1.5 * bm_ratio + noise

# 构建 DataFrame
df_micro = pd.DataFrame({
    'Stock': stocks,
    'B/M': bm_ratio,
    'F-Score': f_score,
    'Future Return (%)': np.round(future_returns, 2),
    'ROA>0': roa_positive,
    'ΔROA>0': delta_roa,
    'CFO>0': cfo_positive,
    'ACCRUE<0': accrue_negative,
    'ΔLEVER<0': delta_lever,
    'ΔLIQUID>0': delta_liquid,
    'No EQ Offer': eq_offer,
    'ΔMARGIN>0': delta_margin,
    'ΔTURN>0': delta_turn
})

print("📋 10 只股票数据集：")
print(df_micro[['Stock', 'B/M', 'F-Score', 'Future Return (%)']].to_string(index=False))
print(f"\n💡 观察：")
print(f"   B/M 最高的股票: S08 (B/M=1.5), F-Score=8")
print(f"   B/M 最低的股票: S09 (B/M=0.2), F-Score=0")
print(f"   高 B/M 不一定意味着高收益，还需要看基本面！")

In [ ]:
# ========== 步骤 1: 按 B/M 排序分组 ==========
print("📊 步骤 1: 按 B/M 排序分组")
print("─" * 55)

df_sorted = df_micro.sort_values('B/M').reset_index(drop=True)

# 分为 2 组：Low B/M (Q1) vs High B/M (Q2)
df_sorted['BM_Group'] = ['Low BM'] * 5 + ['High BM'] * 5

print("\n  Low B/M 组（成长股）:")
for _, row in df_sorted[df_sorted['BM_Group'] == 'Low BM'].iterrows():
    print(f"    {row['Stock']}: B/M = {row['B/M']:.1f}, F-Score = {row['F-Score']}, Return = {row['Future Return (%)']:.2f}%")

print("\n  High B/M 组（价值股）:")
for _, row in df_sorted[df_sorted['BM_Group'] == 'High BM'].iterrows():
    print(f"    {row['Stock']}: B/M = {row['B/M']:.1f}, F-Score = {row['F-Score']}, Return = {row['Future Return (%)']:.2f}%")

# 计算 Spread
ret_high_bm = df_sorted[df_sorted['BM_Group'] == 'High BM']['Future Return (%)'].mean()
ret_low_bm = df_sorted[df_sorted['BM_Group'] == 'Low BM']['Future Return (%)'].mean()
spread_bm = ret_high_bm - ret_low_bm

print(f"\n📊 步骤 2: 计算 B/M Spread")
print(f"  R_High BM = {ret_high_bm:.2f}%")
print(f"  R_Low BM  = {ret_low_bm:.2f}%")
print(f"  Spread = R_High - R_Low = {ret_high_bm:.2f} - {ret_low_bm:.2f} = {spread_bm:.2f}%")
print(f"\n💡 单纯按 B/M 分组，价值股比成长股多赚 {spread_bm:.2f}%")
print(f"   但这包含了价值陷阱！需要结合 F-Score 看看")

In [ ]:
# ========== 步骤 3: 按 F-Score 分组 ==========
print("📊 步骤 3: 按 F-Score 分组（Piotroski 方法）")
print("─" * 55)

# 按 F-Score 分为 3 组：0-3 (Low), 4-6 (Middle), 7-9 (High)
df_micro['F_Group'] = pd.cut(df_micro['F-Score'], 
                              bins=[-1, 3, 6, 9], 
                              labels=['Low (0-3)', 'Middle (4-6)', 'High (7-9)'])

print("\n  F-Score 分组结果：")
for group in ['Low (0-3)', 'Middle (4-6)', 'High (7-9)']:
    group_df = df_micro[df_micro['F_Group'] == group]
    if len(group_df) > 0:
        avg_ret = group_df['Future Return (%)'].mean()
        print(f"\n  {group} 组 ({len(group_df)} 只股票):")
        for _, row in group_df.iterrows():
            print(f"    {row['Stock']}: B/M={row['B/M']:.1f}, F-Score={row['F-Score']}, Return={row['Future Return (%)']:.2f}%")
        print(f"    平均收益 = {avg_ret:.2f}%")

# 计算 High F-Score - Low F-Score Spread
ret_high_f = df_micro[df_micro['F_Group'] == 'High (7-9)']['Future Return (%)'].mean()
ret_low_f = df_micro[df_micro['F_Group'] == 'Low (0-3)']['Future Return (%)'].mean()
spread_f = ret_high_f - ret_low_f

print(f"\n📊 步骤 4: 计算 F-Score Spread")
print(f"  R_High F-Score = {ret_high_f:.2f}%")
print(f"  R_Low F-Score  = {ret_low_f:.2f}%")
print(f"  Spread = {ret_high_f:.2f} - {ret_low_f:.2f} = {spread_f:.2f}%")
print(f"\n💡 F-Score 高的公司确实有更高的未来收益！")

In [ ]:
# ========== 步骤 5: 双重排序：B/M × F-Score ==========
print("📊 步骤 5: 双重排序：B/M × F-Score")
print("─" * 55)

# 按 B/M 分 2 组
df_micro['BM_Group'] = pd.qcut(df_micro['B/M'], 2, labels=['Low BM', 'High BM'])

# 按 F-Score 分 2 组
df_micro['F_Group2'] = pd.cut(df_micro['F-Score'], 
                               bins=[-1, 4, 9], 
                               labels=['Low F', 'High F'])

# 交叉分组
cross_table = df_micro.groupby(['BM_Group', 'F_Group2'])['Future Return (%)'].agg(['mean', 'count'])
print("\n  双重排序结果（平均收益 %）：")
print(cross_table.to_string())

# 计算真价值股 vs 价值陷阱
true_value = df_micro[(df_micro['BM_Group'] == 'High BM') & (df_micro['F_Group2'] == 'High F')]
value_trap = df_micro[(df_micro['BM_Group'] == 'High BM') & (df_micro['F_Group2'] == 'Low F')]

print(f"\n💡 核心发现：")
print(f"   真价值股（高 B/M + 高 F-Score）: 平均收益 = {true_value['Future Return (%)'].mean():.2f}%")
print(f"   价值陷阱（高 B/M + 低 F-Score）: 平均收益 = {value_trap['Future Return (%)'].mean():.2f}%")
print(f"   差异 = {true_value['Future Return (%)'].mean() - value_trap['Future Return (%)'].mean():.2f}%")
print(f"\n🎯 结论：高 B/M 不一定赚钱，必须结合 F-Score 筛选基本面好的公司！")

In [ ]:
# ========== 可视化：F-Score 与未来收益 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 图1: F-Score vs 未来收益 ---
ax1 = axes[0]
colors = ['#e74c3c' if fs < 4 else '#f39c12' if fs < 7 else '#2ecc71' for fs in df_micro['F-Score']]
ax1.scatter(df_micro['F-Score'], df_micro['Future Return (%)'], 
            c=colors, s=100, edgecolors='black', alpha=0.8)

# 标注股票名
for i, row in df_micro.iterrows():
    ax1.annotate(row['Stock'], (row['F-Score'], row['Future Return (%)']), 
                 fontsize=9, ha='center', va='bottom')

# 添加趋势线
z = np.polyfit(df_micro['F-Score'], df_micro['Future Return (%)'], 1)
p = np.poly1d(z)
x_line = np.linspace(0, 9, 100)
ax1.plot(x_line, p(x_line), 'r--', linewidth=2, alpha=0.7, label=f'Trend: y={z[0]:.2f}x+{z[1]:.2f}')

ax1.set_xlabel('F-Score', fontsize=12)
ax1.set_ylabel('Future Return (%)', fontsize=12)
ax1.set_title('F-Score vs Future Returns', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)
ax1.axhline(y=0, color='black', linewidth=0.8)

# --- 图2: B/M vs 未来收益（按 F-Score 着色）---
ax2 = axes[1]
scatter = ax2.scatter(df_micro['B/M'], df_micro['Future Return (%)'], 
                      c=df_micro['F-Score'], cmap='RdYlGn', s=100, 
                      edgecolors='black', alpha=0.8, vmin=0, vmax=9)
plt.colorbar(scatter, ax=ax2, label='F-Score')

# 标注股票名
for i, row in df_micro.iterrows():
    ax2.annotate(row['Stock'], (row['B/M'], row['Future Return (%)']), 
                 fontsize=9, ha='center', va='bottom')

ax2.set_xlabel('B/M Ratio', fontsize=12)
ax2.set_ylabel('Future Return (%)', fontsize=12)
ax2.set_title('B/M vs Future Returns (colored by F-Score)', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)
ax2.axhline(y=0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"   左图：F-Score 与未来收益正相关，基本面越好的公司收益越高")
print(f"   右图：颜色越绿（F-Score 越高）的点，收益越高")
print(f"   关键：高 B/M 的股票中，只有 F-Score 高的才能获得正收益")

---

## 4. 完整流程：300 只股票 × 60 个月

### 🎯 完整异象检验

现在我们**从头开始**构建完整的估值异象检验：

1. 模拟 300 只股票、60 个月的截面数据
2. 每月计算 F-Score 和 B/M
3. 按 B/M × F-Score 双重排序 → 构建多空组合
4. 收集 60 个月的 Spread
5. 将 Spread 对 FF3 做回归 → 检验 Alpha

### 📐 DGP (数据生成过程)

$$r_{i,t} = \beta_{\text{MKT},i} \cdot \text{MKT}_t + \beta_{\text{SMB},i} \cdot \text{SMB}_t + \beta_{\text{HML},i} \cdot \text{HML}_t + \gamma_{\text{F}} \cdot \text{F-Score}_{i,t} + \varepsilon_{i,t}$$

其中：
- $\text{MKT}_t \sim N(0.5, 4^2)$, $\text{SMB}_t \sim N(0.2, 2^2)$, $\text{HML}_t \sim N(0.3, 2^2)$
- $\beta_{\text{MKT},i} \sim U(0.5, 1.5)$, $\beta_{\text{SMB},i} \sim U(-0.5, 0.5)$, $\beta_{\text{HML},i} \sim U(-0.5, 0.5)$
- $\text{F-Score}_{i,t} \sim \text{Binomial}(9, 0.5)$（0-9 分）
- $\gamma_{\text{F}} = 0.3$（F-Score 的真实效应）
- $\varepsilon_{i,t} \sim N(0, 5^2)$（个股噪声）

In [ ]:
# ========== 完整 DGP: 300 只股票 × 60 个月 ==========
np.random.seed(42)
N_STOCKS = 300
N_MONTHS = 60
N_GROUPS = 5

# FF3 因子月度收益
MKT_f = np.random.normal(0.5, 4, N_MONTHS)
SMB_f = np.random.normal(0.2, 2, N_MONTHS)
HML_f = np.random.normal(0.3, 2, N_MONTHS)

# 每只股票的因子暴露（固定不变）
beta_mkt_i = np.random.uniform(0.5, 1.5, N_STOCKS)
beta_smb_i = np.random.uniform(-0.5, 0.5, N_STOCKS)
beta_hml_i = np.random.uniform(-0.5, 0.5, N_STOCKS)

# F-Score 的真实效应系数
gamma_fscore = 0.3  # 每增加 1 分 F-Score → 未来收益增加 0.3%

# 存储每月的 Spread
valuation_spreads = []  # 纯 B/M Spread
fscore_spreads = []     # F-Score Spread
combined_spreads = []   # B/M + F-Score 组合 Spread

print(f"📊 模拟参数：")
print(f"  股票数量: {N_STOCKS} 只/月")
print(f"  时间跨度: {N_MONTHS} 个月")
print(f"  分组数量: {N_GROUPS} 组")
print(f"  FF3 因子: MKT~N(0.5,4²), SMB~N(0.2,2²), HML~N(0.3,2²)")
print(f"  F-Score 效应: γ = {gamma_fscore}")
print(f"")
print(f"🔄 开始逐月模拟...")

for t in range(N_MONTHS):
    # 本月 F-Score（截面变异）
    fscore = np.random.binomial(9, 0.5, N_STOCKS)  # 0-9 分
    
    # 本月 B/M（与 F-Score 负相关，模拟价值陷阱）
    bm = np.random.lognormal(0, 0.5, N_STOCKS)
    bm = bm * (1 - 0.05 * fscore)  # F-Score 低的公司 B/M 可能更高
    
    # 噪声
    eps = np.random.normal(0, 5, N_STOCKS)
    
    # 收益率 = 因子暴露 × 因子收益 + F-Score 效应 + 噪声
    ret = (beta_mkt_i * MKT_f[t] + beta_smb_i * SMB_f[t]
           + beta_hml_i * HML_f[t] + gamma_fscore * fscore + eps)
    
    # --- 策略 1: 纯 B/M 排序 ---
    df_t = pd.DataFrame({'bm': bm, 'ret': ret, 'fscore': fscore})
    df_t['bm_group'] = pd.qcut(df_t['bm'], N_GROUPS, labels=[f'Q{i}' for i in range(1, N_GROUPS+1)])
    bm_group_ret = df_t.groupby('bm_group')['ret'].mean()
    spread_bm = bm_group_ret['Q5'] - bm_group_ret['Q1']  # 高 B/M - 低 B/M
    valuation_spreads.append(spread_bm)
    
    # --- 策略 2: 纯 F-Score 排序 ---
    df_t['f_group'] = pd.cut(df_t['fscore'], bins=[-1, 3, 6, 9], labels=['Low', 'Mid', 'High'])
    f_group_ret = df_t.groupby('f_group')['ret'].mean()
    spread_f = f_group_ret['High'] - f_group_ret['Low']
    fscore_spreads.append(spread_f)
    
    # --- 策略 3: B/M + F-Score 组合 ---
    # 在高 B/M 组中，按 F-Score 分组
    high_bm = df_t[df_t['bm_group'] == 'Q5']
    if len(high_bm) > 5:
        high_bm['f_group2'] = pd.cut(high_bm['fscore'], bins=[-1, 4, 9], labels=['Low F', 'High F'])
        combined_ret = high_bm.groupby('f_group2')['ret'].mean()
        if 'High F' in combined_ret.index and 'Low F' in combined_ret.index:
            spread_combined = combined_ret['High F'] - combined_ret['Low F']
            combined_spreads.append(spread_combined)
    
    if t < 3:
        print(f"\n  月份 {t+1}:")
        print(f"    纯 B/M Spread (Q5-Q1) = {spread_bm:.2f}%")
        print(f"    F-Score Spread (High-Low) = {spread_f:.2f}%")
        if len(combined_spreads) > t:
            print(f"    组合 Spread (High F - Low F in High BM) = {combined_spreads[-1]:.2f}%")

valuation_spreads = np.array(valuation_spreads)
fscore_spreads = np.array(fscore_spreads)
combined_spreads = np.array(combined_spreads)

print(f"\n  ... (省略第 4~{N_MONTHS} 月) ...")
print(f"\n✅ 完成！")
print(f"  纯 B/M Spread 均值: {valuation_spreads.mean():.3f}%")
print(f"  F-Score Spread 均值: {fscore_spreads.mean():.3f}%")
print(f"  组合 Spread 均值: {combined_spreads.mean():.3f}%")

In [ ]:
# ========== 异象检验: 对 FF3 做回归 ==========
print("═" * 60)
print("📋 异象检验: 回归 Spread 对 FF3 因子")
print("═" * 60)

# 构建 FF3 自变量矩阵
X_ff3 = sm.add_constant(np.column_stack([MKT_f, SMB_f, HML_f]))
factor_names = ['Alpha', 'MKT', 'SMB', 'HML']

# ---- 策略 1: 纯 B/M ----
print("\n" + "─" * 60)
print("📊 检验 1: 纯 B/M 排序")
print("─" * 60)

model_bm = sm.OLS(valuation_spreads, X_ff3).fit()

print(f"\n  {'参数':>10s}  {'估计值':>10s}  {'标准误':>10s}  {'t值':>10s}  {'P值':>10s}  {'显著?':>6s}")
print(f"  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*6}")
for name, coef, se, tv, pv in zip(factor_names, model_bm.params, model_bm.bse,
                                   model_bm.tvalues, model_bm.pvalues):
    sig = '✅' if abs(tv) > 2 else ''
    print(f"  {name:>10s}  {coef:>10.4f}  {se:>10.4f}  {tv:>10.4f}  {pv:>10.6f}  {sig:>6s}")

print(f"\n  R² = {model_bm.rsquared:.4f}")
print(f"  🎯 Alpha = {model_bm.params[0]:.4f}%, t = {model_bm.tvalues[0]:.4f}")
if abs(model_bm.tvalues[0]) > 2:
    print(f"  ✅ Alpha 显著 → 纯 B/M 排序构成异象")
else:
    print(f"  ❌ Alpha 不显著 → 纯 B/M 排序不构成异象")

# ---- 策略 2: F-Score ----
print("\n" + "─" * 60)
print("📊 检验 2: F-Score 排序")
print("─" * 60)

model_f = sm.OLS(fscore_spreads, X_ff3).fit()

print(f"\n  {'参数':>10s}  {'估计值':>10s}  {'标准误':>10s}  {'t值':>10s}  {'P值':>10s}  {'显著?':>6s}")
print(f"  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*6}")
for name, coef, se, tv, pv in zip(factor_names, model_f.params, model_f.bse,
                                   model_f.tvalues, model_f.pvalues):
    sig = '✅' if abs(tv) > 2 else ''
    print(f"  {name:>10s}  {coef:>10.4f}  {se:>10.4f}  {tv:>10.4f}  {pv:>10.6f}  {sig:>6s}")

print(f"\n  R² = {model_f.rsquared:.4f}")
print(f"  🎯 Alpha = {model_f.params[0]:.4f}%, t = {model_f.tvalues[0]:.4f}")
if abs(model_f.tvalues[0]) > 2:
    print(f"  ✅ Alpha 显著 → F-Score 排序构成异象")
else:
    print(f"  ❌ Alpha 不显著 → F-Score 排序不构成异象")

# ---- 策略 3: 组合 ----
print("\n" + "─" * 60)
print("📊 检验 3: B/M + F-Score 组合（在高 B/M 中选高 F-Score）")
print("─" * 60)

model_combined = sm.OLS(combined_spreads, X_ff3[:len(combined_spreads)]).fit()

print(f"\n  {'参数':>10s}  {'估计值':>10s}  {'标准误':>10s}  {'t值':>10s}  {'P值':>10s}  {'显著?':>6s}")
print(f"  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*6}")
for name, coef, se, tv, pv in zip(factor_names, model_combined.params, model_combined.bse,
                                   model_combined.tvalues, model_combined.pvalues):
    sig = '✅' if abs(tv) > 2 else ''
    print(f"  {name:>10s}  {coef:>10.4f}  {se:>10.4f}  {tv:>10.4f}  {pv:>10.6f}  {sig:>6s}")

print(f"\n  R² = {model_combined.rsquared:.4f}")
print(f"  🎯 Alpha = {model_combined.params[0]:.4f}%, t = {model_combined.tvalues[0]:.4f}")
if abs(model_combined.tvalues[0]) > 2:
    print(f"  ✅ Alpha 显著 → 组合策略构成异象")
else:
    print(f"  ❌ Alpha 不显著 → 组合策略不构成异象")

In [ ]:
# ========== 可视化: 三种策略对比 ==========
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
months = np.arange(1, N_MONTHS + 1)

# --- 图1: 纯 B/M Spread 时间序列 ---
ax1 = axes[0, 0]
ax1.bar(months, valuation_spreads,
        color=['#2ecc71' if s > 0 else '#e74c3c' for s in valuation_spreads],
        alpha=0.7, edgecolor='none')
ax1.axhline(y=valuation_spreads.mean(), color='blue', linestyle='--', linewidth=2,
            label=f'Mean = {valuation_spreads.mean():.2f}%')
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Month', fontsize=11)
ax1.set_ylabel('Spread (%)', fontsize=11)
ax1.set_title('Pure B/M Sort: Monthly Spread', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# --- 图2: F-Score Spread 时间序列 ---
ax2 = axes[0, 1]
ax2.bar(months, fscore_spreads,
        color=['#2ecc71' if s > 0 else '#e74c3c' for s in fscore_spreads],
        alpha=0.7, edgecolor='none')
ax2.axhline(y=fscore_spreads.mean(), color='blue', linestyle='--', linewidth=2,
            label=f'Mean = {fscore_spreads.mean():.2f}%')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_xlabel('Month', fontsize=11)
ax2.set_ylabel('Spread (%)', fontsize=11)
ax2.set_title('F-Score Sort: Monthly Spread', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

# --- 图3: 组合 Spread 时间序列 ---
ax3 = axes[1, 0]
months_combined = np.arange(1, len(combined_spreads) + 1)
ax3.bar(months_combined, combined_spreads,
        color=['#2ecc71' if s > 0 else '#e74c3c' for s in combined_spreads],
        alpha=0.7, edgecolor='none')
ax3.axhline(y=combined_spreads.mean(), color='blue', linestyle='--', linewidth=2,
            label=f'Mean = {combined_spreads.mean():.2f}%')
ax3.axhline(y=0, color='black', linewidth=0.8)
ax3.set_xlabel('Month', fontsize=11)
ax3.set_ylabel('Spread (%)', fontsize=11)
ax3.set_title('Combined (High BM + High F-Score): Monthly Spread', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(axis='y', alpha=0.3)

# --- 图4: Alpha 对比 ---
ax4 = axes[1, 1]
strategies = ['Pure B/M', 'F-Score', 'Combined']
alphas = [model_bm.params[0], model_f.params[0], model_combined.params[0]]
t_values = [model_bm.tvalues[0], model_f.tvalues[0], model_combined.tvalues[0]]
colors_alpha = ['#2ecc71' if abs(t) > 2 else '#95a5a6' for t in t_values]

bars = ax4.bar(strategies, alphas, color=colors_alpha, edgecolor='black', alpha=0.85, width=0.5)
ax4.axhline(y=0, color='red', linestyle='--', linewidth=1.5, label='H₀: α=0')

# 标注 t 值
for bar, a, tv in zip(bars, alphas, t_values):
    va = 'bottom' if a >= 0 else 'top'
    offset = 0.05 if a >= 0 else -0.05
    ax4.text(bar.get_x() + bar.get_width()/2., a + offset,
             f'α={a:.2f}%\nt={tv:.2f}', ha='center', va=va, fontweight='bold', fontsize=10)

ax4.set_ylabel('Alpha (%)', fontsize=12)
ax4.set_title('Alpha Comparison: Three Strategies', fontsize=13, fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  上排：三种策略的月度 Spread 时间序列")
print(f"  下排左：组合策略的 Spread（在高 B/M 中选高 F-Score）")
print(f"  下排右：三种策略的 Alpha 对比（绿色 = 显著，灰色 = 不显著）")
print(f"  🎯 关键发现：F-Score 能帮助识别真正的价值股，避免价值陷阱")

In [ ]:
# ========== 综合对比报告 ==========
print("=" * 60)
print("📋 估值异象检验综合报告")
print("=" * 60)

print(f"\n🎯 研究问题:")
print(f"   估值指标（B/M）能否预测未来收益？F-Score 能否提升预测能力？")

print(f"\n📊 数据概况:")
print(f"   股票数量: {N_STOCKS} 只/月")
print(f"   时间跨度: {N_MONTHS} 个月")
print(f"   分组方式: {N_GROUPS} 分位，Q5-Q1 Spread")

print(f"\n📊 Spread 统计:")
print(f"   {'策略':>15s}  {'Spread均值':>12s}  {'Spread标准差':>14s}  {'Sharpe Ratio':>14s}")
print(f"   {'─'*15}  {'─'*12}  {'─'*14}  {'─'*14}")

sr_bm = valuation_spreads.mean() / valuation_spreads.std(ddof=1) * np.sqrt(12)
sr_f = fscore_spreads.mean() / fscore_spreads.std(ddof=1) * np.sqrt(12)
sr_combined = combined_spreads.mean() / combined_spreads.std(ddof=1) * np.sqrt(12)

print(f"   {'纯 B/M':>15s}  {valuation_spreads.mean():>12.3f}%  {valuation_spreads.std(ddof=1):>14.3f}%  {sr_bm:>14.3f}")
print(f"   {'F-Score':>15s}  {fscore_spreads.mean():>12.3f}%  {fscore_spreads.std(ddof=1):>14.3f}%  {sr_f:>14.3f}")
print(f"   {'组合':>15s}  {combined_spreads.mean():>12.3f}%  {combined_spreads.std(ddof=1):>14.3f}%  {sr_combined:>14.3f}")

print(f"\n🧮 Alpha 检验 (控制 FF3 因子):")
print(f"   {'策略':>15s}  {'α̂':>10s}  {'t_α':>10s}  {'P值':>12s}  {'结论':>10s}")
print(f"   {'─'*15}  {'─'*10}  {'─'*10}  {'─'*12}  {'─'*10}")

for name, model in [('纯 B/M', model_bm), ('F-Score', model_f), ('组合', model_combined)]:
    alpha = model.params[0]
    t_alpha = model.tvalues[0]
    p_alpha = model.pvalues[0]
    conclusion = '✅ 异象' if abs(t_alpha) > 2 else '❌ 非异象'
    print(f"   {name:>15s}  {alpha:>10.4f}%  {t_alpha:>10.4f}  {p_alpha:>12.6f}  {conclusion:>10s}")

print(f"\n🎯 结论:")
print(f"  1. 纯 B/M 排序可能产生显著 Alpha，但包含价值陷阱")
print(f"  2. F-Score 能有效识别基本面好的公司")
print(f"  3. 组合策略（高 B/M + 高 F-Score）能获得更稳定的 Alpha")
print(f"  4. 价值投资 = 估值因子 + 基本面筛选")

print("\n" + "=" * 60)

---

## 📌 核心概念回顾

### 📌 估值异象 (Valuation Anomaly)
- **定义**: 估值指标（B/M, E/P 等）能预测未来收益的现象
- **公式**: $R_{i,t+1} = \alpha + \beta \cdot \text{B/M}_{i,t} + \varepsilon_{i,t}$
- **含义**: 低估值股票（高 B/M）长期来看有更高的预期收益
- **局限**: 包含价值陷阱，需要结合基本面筛选

### 📌 F-Score (Piotroski 评分)
- **定义**: 9 个指标的综合评分（0-9 分），衡量公司基本面强弱
- **公式**: $\text{F-Score} = \sum_{k=1}^{9} I_k$，每个指标满足条件得 1 分
- **含义**: F-Score 越高，公司基本面越健康
- **判断标准**: 0-3 分（弱）、4-6 分（中）、7-9 分（强）

### 📌 价值投资 vs 价值因子
- **价值因子**: 单纯按估值指标选股，简单但容易选出伪价值股
- **价值投资**: 估值因子 + 基本面筛选，找到真正便宜的好公司
- **核心差异**: 是否考虑公司基本面质量

### 📌 异象检验方法
- **排序分组**: 按变量排序 → 构建多空组合 → 计算 Spread
- **Alpha 检验**: 将 Spread 对因子模型回归 → 检验 α 是否显著
- **判断标准**: |t_α| > 2 → Alpha 显著 → 异象成立

### 🔗 完整流程
```
计算估值指标 → 计算 F-Score → 双重排序分组 → 构建多空组合
      ↓              ↓              ↓              ↓
   B/M, E/P      9个指标        高B/M×高F      Long-Short
      ↓              ↓              ↓              ↓
   收集 Spread → 对 FF3 回归 → 检验 Alpha → 判断异象
```

---

## ❌ 常见误区

### ❌ 误区 1: 低 PE/PB 的股票就是好股票
**✓ 正确理解**: 低估值可能是价值陷阱，公司基本面可能在恶化。必须结合 F-Score 等基本面指标筛选。

### ❌ 误区 2: F-Score 越高的股票收益一定越高
**✓ 正确理解**: F-Score 衡量的是基本面质量，但收益还受估值、市场情绪等因素影响。高 F-Score + 低估值才是最佳组合。

### ❌ 误区 3: 价值投资就是买低 PE/PB 的股票
**✓ 正确理解**: 价值投资 = 价值因子 + 基本面筛选。单纯按估值选股是"公式化价值投资"，容易选出伪价值股。

### ❌ 误区 4: 异象检验只需要看原始 Spread 的 T 检验
**✓ 正确理解**: 必须控制已知因子（如 FF3）后检验 Alpha。原始 Spread 显著可能只是因为暴露了已知因子。

### ❌ 误区 5: F-Score 的 9 个指标权重相同
**✓ 正确理解**: Piotroski (2000) 假设每个指标等权重，但不同指标的重要性可能不同。实际应用中可以根据数据调整权重。